# Simulador de Carrera: ProTour FETRI 2026
Este notebook te permite ejecutar y analizar la simulación del ranking predictivo para una competición específica, paso a paso.

In [1]:
from dotenv import load_dotenv
load_dotenv()

import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

import pandas as pd
from chatbot.database_service import get_active_athletes, get_athletes_history_batch
from chatbot.chatbot_api import build_athlete_features
from chatbot.shap_explainer import get_explainer

# 1. Obtener datos de la BD
print("Cargando atletas...")
athletes = get_active_athletes()

print("Cargando historiales...")
history_map = get_athletes_history_batch()

c:\Users\mario\PycharmProjects\conersys_predictivo_triatlon_elitemasculino\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando atletas...
Cargando historiales...


In [2]:
# 2. Cargar Explainer (Motor de Machine Learning)
print("Inicializando explainer...")
explainer = get_explainer()

Inicializando explainer...
[SHAP] Modelo local v2 cargado desde cache.
[SHAP] TreeExplainer inicializado.


In [3]:
# 3. Configurar condiciones de carrera: ProTour FETRI 2026
race_overrides = {
    "race_city": "Madrid",
    "race_temperature": 32.0,
    "race_altitude_m": 650.0,
    "race_humidity": 35.0,
    "avg_ranking_opponents": 50.0,  # Contexto REAL de la carrera
    "n_participants_race": 40       # El ProTour solo tiene a los mejores
}

# 3.5 Calcular puntos totales para sacar el Ranking Real FETRI
for ath in athletes:
    doc = ath["documento"]
    hist = history_map.get(doc, [])
    puntos = sorted([h.get("puntos", 0) for h in hist if h.get("puntos") is not None], reverse=True)[:3]
    ath["total_puntos"] = sum(puntos)
    
athletes.sort(key=lambda x: x["total_puntos"], reverse=True)
for i, ath in enumerate(athletes):
    if ath["total_puntos"] > 0:
        ath["ranking_real"] = float(i + 1)
    else:
        # Novatos absolutos o sin ranking local. Para una simulación Élite, su ranking es bajo
        ath["ranking_real"] = 2500.0

# 4. Generar dataset de features
print(f"Generando variables para {len(athletes)} atletas...")
features_list = []
docs_list = []
for ath in athletes:
    doc = ath["documento"]
    history = history_map.get(doc, [])
    features = build_athlete_features(ath, history, race_overrides=race_overrides)
    features_list.append(features)
    docs_list.append((doc, ath["nombre"], ath["club"]))

print("\nEjemplo de features (variables) enviadas a la IA para el primer atleta:")
pd.DataFrame([features_list[0]]).T


Generando variables para 1028 atletas...

Ejemplo de features (variables) enviadas a la IA para el primer atleta:


,0
athlete_club,UNIVERSIDAD DE ALICANTE
athlete_federation,MAD
athlete_ranking_triatlon,1.0
form_score,97.3
avg_ranking_opponents,50.0
pct_opponents_ranked_higher,0.0
n_participants_race,40
n_races_before,4
n_elite_races_before,4
avg_finish_pos_last_3,2.67


In [4]:
# 5. Predicción en bloque contra Azure ML
print("Consultando el modelo (Azure ML o Local)...")
predictions = explainer.predict_batch(features_list)

# 6. Procesar resultados
results = []
for i, (doc, nombre, club) in enumerate(docs_list):
    results.append({
        "documento": doc,
        "nombre": nombre,
        "club": club,
        "prediccion_algoritmo": predictions[i]
    })

results.sort(key=lambda x: x["prediccion_algoritmo"])
df_results = pd.DataFrame(results)
df_results.insert(0, "ranking_simulado", range(1, len(df_results) + 1))

# Mostrar el Top 20
df_results.head(20)

Consultando el modelo (Azure ML o Local)...


,ranking_simulado,documento,nombre,club,prediccion_algoritmo
0,1,49213350C,ELOY CANALES ESCRIBANO,UNIVERSIDAD DE ALICANTE,29.207154
1,2,18456150F,IZAN EDO AGUILAR,UNIVERSIDAD DE ALICANTE,29.742751
2,3,49572060E,DAVID CANTERO DEL CAMPO,UNIVERSIDAD DE ALICANTE,30.287640
3,4,44067533Q,JESUS VELA VELA,PEÑOTA DENTAL ALUSIGMA,31.286164
4,5,50577591Q,PELAYO GONZÁLEZ TÚRREZ,PEÑOTA DENTAL ALUSIGMA,31.328614
5,6,34278366V,ESTEBAN BASANTA FOUZ,CIDADE DE LUGO FLUVIAL,31.878564
6,7,48787474N,EDUARDO BLANCO MOLLA,UNIVERSIDAD DE ALICANTE,31.925375
7,8,47735423G,FRANCESC GODOY CONTRERAS,SALTOKI TRIKIDEAK,32.320220
8,9,40363410C,GENIS GRAU PLANA,CIDADE DE LUGO FLUVIAL,32.559326
9,10,13392167A,ANDRES EDUARDO DIAZ MARIN,CHAMO DIAZ,32.596843
